# 04 — Architecture web : API, frontend, backend

**Objectif :** comprendre **comment s'assemble une application web moderne** et reconnaître les pièces du puzzle. Indispensable avant d'aller plus loin avec les agents IA, qui sont souvent intégrés dans une appli web existante.

Comme le notebook 03, c'est volontairement **factuel, droit au but, avec des schémas**. Pas de code à exécuter — on construira une vraie petite app dans un notebook ultérieur du parcours.

À la fin, vous saurez :
- ce qu'est un client, un serveur, une API ;
- la différence frontend / backend / base de données ;
- les 4 styles d'API courants : **REST**, **GraphQL**, **gRPC**, **WebSocket** ;
- les 3 grands styles d'architecture : **monolithe**, **microservices**, **serverless** ;
- les patterns *SPA / SSR / static*, *BFF*, *CDN*, *reverse proxy*, *load balancer* ;
- comment un **agent IA** s'insère là-dedans.

## 1. Le modèle client-serveur

Toute appli web repose sur le même modèle de base : un **client** envoie une requête, un **serveur** renvoie une réponse.

```
   👤 Utilisateur
       │
       ▼
  ┌─────────────────┐                                   ┌──────────────────┐
  │   CLIENT        │  ── HTTPS request ─────────────►  │   SERVEUR        │
  │   (navigateur,  │                                   │   (App Service,  │
  │    appli mobile,│  ◄─── HTTPS response ─────────── │    Functions,    │
  │    SDK Python)  │                                   │    Container)    │
  └─────────────────┘                                   └──────────────────┘
        UI / vues                                          logique métier
                                                           + accès données
```

Une **requête HTTP** contient :
- une **méthode** (`GET`, `POST`, `PUT`, `DELETE`, `PATCH`),
- une **URL** (`/api/users/42`),
- des **headers** (`Authorization: Bearer ...`, `Content-Type: application/json`),
- optionnellement un **body** (JSON la plupart du temps).

Une **réponse HTTP** contient :
- un **code de statut** (200 OK, 201 Created, 400 Bad Request, 401 Unauthorized, 404 Not Found, 500 Server Error…),
- des **headers**,
- un **body** (JSON, HTML, image, etc.).

### Les codes de statut à connaître

| Code | Famille | Signification |
|------|---------|---------------|
| **2xx** | Succès | 200 OK, 201 Created, 204 No Content |
| **3xx** | Redirection | 301 Moved, 304 Not Modified (cache) |
| **4xx** | Erreur client | 400 Bad Request, 401 Unauthorized, 403 Forbidden, 404 Not Found, 429 Too Many Requests |
| **5xx** | Erreur serveur | 500 Internal Server Error, 502 Bad Gateway, 503 Service Unavailable |

## 2. Frontend, backend, base de données — les 3 couches

Une appli web typique a **3 couches**. Comprendre qui fait quoi est essentiel.

```
  ┌────────────────────────────────────────────────────────────────┐
  │  FRONTEND  (ce que l'utilisateur voit)                          │
  │  → HTML / CSS / JavaScript                                      │
  │  → React, Vue, Angular, Svelte, Next.js, Blazor…                │
  │  → tourne dans le NAVIGATEUR (ou app mobile)                    │
  └───────────────────────────┬────────────────────────────────────┘
                              │   HTTPS (API REST/GraphQL/…)
                              ▼
  ┌────────────────────────────────────────────────────────────────┐
  │  BACKEND  (la logique métier)                                   │
  │  → Node.js, Python (FastAPI/Django), .NET, Java (Spring), Go…   │
  │  → expose une API                                                │
  │  → tourne sur un SERVEUR (App Service, Functions, AKS…)         │
  └───────────────────────────┬────────────────────────────────────┘
                              │
                              ▼
  ┌────────────────────────────────────────────────────────────────┐
  │  STOCKAGE  (les données persistantes)                           │
  │  → BDD SQL (Azure SQL, PostgreSQL)                              │
  │  → BDD NoSQL (Cosmos DB, MongoDB)                               │
  │  → Cache (Redis)                                                │
  │  → Fichiers (Blob Storage)                                      │
  │  → Recherche (Azure AI Search, Elasticsearch)                   │
  └────────────────────────────────────────────────────────────────┘
```

### Qui fait quoi ?

| Couche | Responsabilité |
|--------|----------------|
| **Frontend** | Affichage, ergonomie, interactions, validations rapides côté client. Ne fait **pas** confiance aux données qu'il envoie. |
| **Backend** | Validation **sérieuse**, règles métier, autorisation, appels à la BDD ou à d'autres services (Foundry, etc.). C'est lui le **gardien de la cohérence**. |
| **Stockage** | Persister les données. Optimisé pour les requêtes. |

> 🔐 **Règle d'or sécurité :** *« Never trust the client. »* Toute validation de sécurité (auth, autorisation, anti-injection) doit être **côté backend**. Le frontend, c'est juste l'UX.

## 3. Les APIs — le contrat entre frontend et backend

Une **API** *(Application Programming Interface)* expose des **endpoints** que le frontend (ou un autre service) peut appeler.

### 3.1 REST — le standard de fait

REST = **Re**presentational **S**tate **T**ransfer. C'est une convention : chaque entité métier = une **ressource**, identifiée par une **URL**, manipulée par les **méthodes HTTP**.

Exemple — gestion d'utilisateurs :

| Verbe | URL | Action |
|-------|-----|--------|
| `GET` | `/api/users` | Lister tous les utilisateurs |
| `GET` | `/api/users/42` | Lire l'utilisateur 42 |
| `POST` | `/api/users` | Créer un utilisateur |
| `PUT` | `/api/users/42` | Remplacer entièrement l'utilisateur 42 |
| `PATCH` | `/api/users/42` | Modifier partiellement l'utilisateur 42 |
| `DELETE` | `/api/users/42` | Supprimer l'utilisateur 42 |

**Documentation** : avec **OpenAPI** (ex-Swagger), on décrit son API en YAML/JSON → on génère doc + clients. La plupart des Foundry tools utilisent OpenAPI.

### 3.2 GraphQL — un client qui choisit ce qu'il veut

Un seul endpoint (`/graphql`), le client envoie une **query** décrivant exactement ce qu'il veut.

```graphql
query {
  user(id: 42) {
    name
    email
    posts(last: 3) { title }
  }
}
```

👍 Utile quand on a beaucoup de relations entre entités et qu'on veut éviter le sur-fetching. 👎 Plus complexe à mettre en place.

### 3.3 gRPC — performance et contrats stricts

Protocole binaire (Protobuf) sur HTTP/2. Très utilisé en **interne** entre microservices. Pas dans le navigateur (sans proxy).

### 3.4 WebSocket — connexion bidirectionnelle persistante

REST = requête / réponse. **WebSocket** = canal ouvert dans les deux sens. Indispensable pour : chat temps réel, streaming d'un LLM, jeu, notifications push.

Variante moderne : **Server-Sent Events (SSE)** — flux unidirectionnel serveur → client. C'est ce que renvoie l'API OpenAI Responses quand on stream les tokens d'un LLM.

### Quoi choisir ?

| Cas | Style |
|-----|-------|
| Appli web/mobile standard | **REST** + OpenAPI |
| Mobile avec beaucoup de relations / saving bandwidth | **GraphQL** |
| Comm interne entre microservices, latence critique | **gRPC** |
| Streaming de tokens LLM, chat temps réel | **SSE** / **WebSocket** |

## 4. Frontend — c'est quoi, concrètement ?

Le **frontend**, c'est **tout ce que l'utilisateur voit et avec quoi il interagit**. Le bouton sur lequel il clique, le tableau qui s'affiche, le formulaire qu'il remplit. C'est l'interface.

Techniquement, dans une appli web, le frontend tourne **dans le navigateur** (Chrome, Firefox, Safari, Edge). Le serveur envoie du code que le navigateur exécute.

### 4.1 Les trois briques de base du web

Toute page web repose sur **3 langages**, depuis 1995. Aucun framework n'y échappe — ils produisent tous au final ces 3 choses :

| Langage | Rôle | Exemple |
|---------|------|---------|
| **HTML** (HyperText Markup Language) | La **structure** du contenu (paragraphes, titres, listes, formulaires) | `<h1>Bonjour</h1>` |
| **CSS** (Cascading Style Sheets) | Le **style** : couleurs, polices, tailles, espacement, animations | `h1 { color: red; font-size: 32px; }` |
| **JavaScript** (JS) | Le **comportement** : réagir aux clics, charger des données, modifier la page sans la recharger | `button.onclick = () => alert('clic!')` |

**Métaphore** : HTML = squelette, CSS = peau et habits, JS = muscles et cerveau.

Tu peux faire un site complet avec juste HTML+CSS, mais dès que tu veux quelque chose de dynamique (login, panier, dashboard, chat…), il te faut JavaScript.

### 4.2 Une page web minimale, à la main

```html
<!DOCTYPE html>
<html lang="fr">
<head>
  <meta charset="utf-8" />
  <title>Mon premier site</title>
  <style>
    body { font-family: sans-serif; padding: 20px; }
    button { padding: 10px 20px; background: blue; color: white; border: none; }
  </style>
</head>
<body>
  <h1>Bonjour 👋</h1>
  <p>Clique sur le bouton :</p>
  <button id="btn">Clique-moi</button>
  <script>
    document.getElementById('btn').onclick = () => alert('Salut !');
  </script>
</body>
</html>
```

Tu sauves ça en `index.html`, tu ouvres dans Chrome → tu as un site web. C'est tout.

**Le problème** : faire un Gmail ou un Notion à la main en HTML/CSS/JS, c'est ingérable (1 000+ composants, état partagé, performance, accessibilité, tests…). D'où les frameworks.

### 4.3 Les frameworks JavaScript — pourquoi ils existent

Un **framework frontend** te donne :
- Un système de **composants** réutilisables (`<UserCard user={...} />`).
- Une gestion du **state** (l'état de l'app : utilisateur connecté, panier, etc.).
- Un mécanisme pour **mettre à jour l'UI** automatiquement quand le state change.
- Un **routing** (URL → page) côté client.
- Un écosystème (formulaires, animations, i18n, tests).

**Les 4 grands frameworks aujourd'hui** :

| Framework | Créé par | Force | Notre choix dans `mailroom-ia` |
|-----------|----------|-------|-------------------------------|
| **React** ⭐ | Meta (Facebook) | Le plus utilisé au monde, énorme écosystème, basé sur des **composants** + **JSX** (HTML dans JS) | ✅ via Next.js |
| **Vue** | Evan You (indé) | Plus simple à apprendre, syntaxe template proche du HTML | — |
| **Angular** | Google | Très structuré (entreprise lourde), TypeScript obligatoire | — |
| **Svelte** | Rich Harris | Compile le composant en JS optimisé (pas de runtime), très rapide | — |

**Tu ne peux pas te tromper** : tous les 4 font le même boulot. **React** domine en France et en startup, on apprend React → tu trouveras du boulot.

### 4.4 Composant React en 10 lignes

```tsx
// Un composant = une fonction qui renvoie du HTML (JSX)
function Bouton({ texte, onClick }: { texte: string; onClick: () => void }) {
  return (
    <button
      onClick={onClick}
      className="rounded bg-blue-600 px-4 py-2 text-white"
    >
      {texte}
    </button>
  );
}

// Utilisation :
<Bouton texte="Sauvegarder" onClick={() => sauvegarder()} />
```

Le composant est **réutilisable**, **typé** (TypeScript), **stylé** (Tailwind), et son comportement est explicite. C'est la brique de base de React.

### 4.5 Les trois modes de rendu — SPA, SSR, SSG

Quand tu charges `https://mon-site.com/dashboard`, qui génère le HTML que ton navigateur affiche ? Il y a **3 réponses possibles**.

#### A. SPA (Single Page Application)

Le serveur t'envoie une page **quasi vide** + un gros fichier JavaScript. Le JS s'exécute dans ton navigateur, va chercher les données via une API, et **fabrique** la page dans ton navigateur.

```
Navigateur                                Serveur
   │                                         │
   │ GET /dashboard                          │
   ├────────────────────────────────────────►│
   │ ← HTML quasi vide + main.js (300 Ko)    │
   │◄────────────────────────────────────────┤
   │                                         │
   │ (JS s'exécute)                          │
   │ GET /api/me, /api/data                  │
   ├────────────────────────────────────────►│
   │ ← JSON                                  │
   │◄────────────────────────────────────────┤
   │                                         │
   │ (JS construit la page côté navigateur)  │
   ▼
   ✨ Page affichée
```

- ✅ Très **fluide** une fois chargée (les clics ne rechargent rien, juste un appel API + re-render).
- ✅ Le serveur peut être **purement API** (n'importe quel langage backend).
- ❌ **Premier chargement lent** (gros JS à télécharger + exécuter).
- ❌ **Mauvais pour le SEO** (Google voit une page vide).
- ❌ Casse si l'utilisateur a JavaScript désactivé.

**Bon pour** : dashboards, back-offices, apps internes (notre admin `mailroom-ia` est essentiellement une SPA).

#### B. SSR (Server-Side Rendering)

Le serveur **génère** la page HTML **à chaque visite**, déjà remplie avec les données. Le navigateur reçoit du HTML "prêt à afficher".

```
Navigateur                                Serveur
   │                                         │
   │ GET /dashboard                          │
   ├────────────────────────────────────────►│
   │                                         │ (serveur va chercher en DB,
   │                                         │  rend le HTML avec les données)
   │ ← HTML complet (50 Ko)                  │
   │◄────────────────────────────────────────┤
   ▼
   ✨ Page affichée immédiatement
```

- ✅ **Rapide à l'affichage** (HTML pré-rendu).
- ✅ **SEO-friendly** (Google voit la page complète).
- ❌ Le serveur travaille **à chaque visite** → plus de charge serveur.
- ❌ Plus complexe à coder.

**Bon pour** : sites grand public (e-commerce, médias, blogs avec authent).

#### C. SSG (Static Site Generation)

Les pages HTML sont **pré-générées à la build** (à la compilation), puis servies par un CDN comme des fichiers statiques. Pas de calcul à chaque visite.

```
À la build (1 fois) :
   Build server →  génère page1.html, page2.html, … → upload vers CDN

À l'exécution :
   Navigateur  →  GET page1.html  →  CDN  →  ← fichier statique
```

- ✅ **Ultra rapide** (CDN, pas de calcul).
- ✅ **Ultra peu cher** (juste des fichiers à servir).
- ✅ **Sécurité maximale** (pas de serveur à attaquer).
- ❌ Pas adapté au contenu **par utilisateur** (chacun aurait sa page → impossible à pré-générer).
- ❌ Re-build à chaque changement de contenu (mais build = 30 s pour 100 pages).

**Bon pour** : blogs, docs, marketing, landing pages.

#### Et le mode hybride ?

Les **frameworks fullstack modernes** (Next.js, Nuxt, SvelteKit, Remix) te laissent **choisir page par page** : telle page en SSG, telle autre en SSR, telle autre en SPA. C'est ce qu'on fait dans `mailroom-ia` avec Next.js 15 :
- `/admin/inbox` → **SSR** (Server Component qui query Cosmos à chaque visite).
- `/admin/inbox` (le panneau preview/validate) → **SPA-like** (Client Component avec state local).
- Pages marketing → **SSG** (si on en avait).

### 4.6 Mobile, PWA, native — pour info

Quand on dit "frontend", on pense souvent au navigateur, mais il y a d'autres cibles :

| Cible | Tech | Quand |
|-------|------|-------|
| **Site responsive** (web mobile) | Le même site, design qui s'adapte (Tailwind, media queries) | 95 % des cas |
| **PWA** (Progressive Web App) | Site qu'on peut "installer" sur le téléphone, marche offline | Quand tu veux une icône sur le bureau sans passer par les stores |
| **App native iOS/Android** | Swift/Kotlin (natif), ou React Native / Flutter (cross-platform) | Quand tu as besoin d'accès matériel poussé (caméra, NFC, BLE) ou de présence dans les stores |

Pour `mailroom-ia` : site responsive (Tailwind mobile-first), pas besoin de plus.

### 4.7 Bundlers, transpilation — vite fait

Quand tu écris du React + TypeScript + Tailwind + JSX, **le navigateur ne comprend rien** à tout ça. Il faut un **bundler** qui :

1. **Transpile** : convertit TypeScript → JavaScript, JSX → fonctions, CSS Tailwind → CSS classique.
2. **Bundle** : agrège tes 200 fichiers en 1 ou 2 gros fichiers JS (`main.js`, `vendor.js`).
3. **Minifie** : enlève les espaces et raccourcit les noms → fichier 5× plus petit.
4. **Optimise** : tree-shaking (enlève le code mort), code-splitting (charge à la demande).

Outils standards en 2026 : **Vite** (pour les SPAs purs), **Next.js** / **Nuxt** / **SvelteKit** (incluent leur propre bundler), **Turbopack** (Vercel, prochaine gen).

### 4.8 La stack frontend de `mailroom-ia` — récap

- **Next.js 15** (framework fullstack React) — App Router, Server Components.
- **React 19** (composants).
- **TypeScript** (typage strict).
- **Tailwind CSS** + **shadcn/ui** (style + composants accessibles).
- **lucide-react** (icônes).
- **Zod** (validation runtime des inputs).
- Buildé en image Docker (`node:22-alpine`), servi par un Container App.

> 💡 **Pourquoi Next.js plutôt que React seul ?** Parce que Next.js te donne le routing, le SSR, l'optimisation des images, les API routes (`/api/upload`), et le build/déploiement en une seule commande. Pour notre app admin, c'est le sweet spot.


## 5. Backend — les 3 grands styles d'architecture

### 5.1 Monolithe

Une **seule application**, un **seul déploiement**, une **seule base de données**. Tout est dedans (auth, métier, batch…).

```
  ┌─────────────────────────────────────────────┐
  │              MONOLITHE                      │
  │  ┌──────┐ ┌──────┐ ┌──────┐ ┌────────────┐  │      ┌─────┐
  │  │ Auth │ │ Users│ │Orders│ │ Invoices…  │  │ ───► │ BDD │
  │  └──────┘ └──────┘ └──────┘ └────────────┘  │      └─────┘
  └─────────────────────────────────────────────┘
```

👍 Simple à démarrer, à débugger, à déployer.  
👎 Devient lourd à grossir, un bug bloque tout, scaling difficile.

**À utiliser** : MVP, petites équipes (< 10 devs), démarrage de projet.

### 5.2 Microservices

Plein de petits services indépendants, chacun avec **sa propre BDD**, communiquant par API ou par messages.

```
  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐
  │  Auth    │  │  Users   │  │  Orders  │  │ Invoices │  ← chacun déployé séparément
  │  service │  │  service │  │  service │  │  service │
  └────┬─────┘  └────┬─────┘  └────┬─────┘  └────┬─────┘
       │             │             │             │
       ▼             ▼             ▼             ▼
    ┌────┐        ┌────┐        ┌────┐        ┌────┐
    │BDD │        │BDD │        │BDD │        │BDD │     ← chacun sa BDD
    └────┘        └────┘        └────┘        └────┘
                                                          + un bus de messages
                                                            (Service Bus, Kafka)
```

👍 Scaling indépendant, équipes indépendantes, choix de techno par service.  
👎 Complexité opérationnelle énorme (réseau, observabilité, transactions distribuées).

**À utiliser** : grosses équipes (> 50 devs), produits matures, scaling très différencié par fonctionnalité.

### 5.3 Serverless

On déploie des **fonctions** isolées (Azure Functions, AWS Lambda) qui ne tournent que quand on les appelle. Facturé à l'invocation et au temps d'exécution.

```
  HTTP trigger ───► [ Function: createOrder ]  ──► BDD
  Timer 5 min ────► [ Function: cleanupOldFiles ]
  Queue msg ──────► [ Function: sendInvoiceEmail ]
```

👍 Pay-per-use, scaling auto à zéro, ops minimales.  
👎 *Cold start*, limites de durée (typiquement 10 min), debug parfois pénible.

**À utiliser** : APIs avec trafic en pics, jobs périodiques, webhooks, glue code.

> 💡 La plupart des projets cloud modernes **mixent** : un monolithe ou quelques services pour la logique métier + des Functions pour les jobs/webhooks + du serverless pour les agents IA.

## 6. Les pièces qui orchestrent tout ça en prod

Une appli en production n'est jamais juste « frontend + backend + BDD ». Plusieurs briques s'ajoutent.

```
                              🌍 Internet
                                  │
                                  ▼
                         ┌────────────────┐
                         │      CDN       │   ← cache statique (images, JS, CSS)
                         │  (Front Door,  │      proche de l'utilisateur
                         │   Cloudflare)  │
                         └────────┬───────┘
                                  │
                                  ▼
                         ┌────────────────┐
                         │  Reverse proxy │   ← termine TLS, route les URLs
                         │  (App Gateway, │      ("/api/..." → backend,
                         │   nginx)       │       "/..." → frontend)
                         └────────┬───────┘
                                  │
                                  ▼
                         ┌────────────────┐
                         │ Load Balancer  │   ← distribue le trafic sur N
                         └────────┬───────┘      instances de backend
                                  │
                       ┌──────────┼──────────┐
                       ▼          ▼          ▼
                  ┌──────┐   ┌──────┐   ┌──────┐
                  │  API │   │  API │   │  API │   ← scaling horizontal
                  │  v1  │   │  v1  │   │  v1  │
                  └───┬──┘   └───┬──┘   └───┬──┘
                      │          │          │
                      └──────────┼──────────┘
                                 ▼
                          ┌──────────────┐         ┌─────────────┐
                          │     BDD      │         │   Cache     │
                          │  (Azure SQL, │ ◄────►  │   (Redis)   │
                          │   Cosmos)    │         └─────────────┘
                          └──────────────┘
```

| Brique | Rôle |
|--------|------|
| **CDN** *Content Delivery Network* | Cache statique mondial → l'utilisateur télécharge les assets depuis un POP proche. (Azure Front Door, Cloudflare) |
| **Reverse proxy** | Point d'entrée HTTPS, termine le TLS, route selon l'URL, peut filtrer (WAF). |
| **Load balancer** | Distribue le trafic sur plusieurs instances backend pour scale + tolérance aux pannes. |
| **Cache** | Stocker temporairement des données chères à recalculer (résultats DB, sessions). |

## 7. Le pattern BFF — *Backend For Frontend*

Au lieu qu'un même backend serve plusieurs clients très différents (web + mobile + Teams), on crée **un backend par type de client**, optimisé pour ses besoins.

```
       ┌────────────┐         ┌────────────┐         ┌────────────┐
       │  Web (SPA) │         │  Mobile    │         │  Teams bot │
       └─────┬──────┘         └─────┬──────┘         └─────┬──────┘
             │                       │                       │
             ▼                       ▼                       ▼
       ┌────────────┐         ┌────────────┐         ┌────────────┐
       │   BFF Web  │         │ BFF Mobile │         │ BFF Teams  │  ← agrégateurs spécifiques
       └─────┬──────┘         └─────┬──────┘         └─────┬──────┘
             │                       │                       │
             └───────────────────────┼───────────────────────┘
                                     ▼
                           ┌──────────────────────┐
                           │  Services métier     │  ← partagés, réutilisables
                           │  (Users, Orders, …)  │
                           └──────────────────────┘
```

Très utilisé quand on a une **agrégation** d'appels (1 vue mobile = 4 calls services métier → le BFF fait les 4 et renvoie 1 JSON adapté).

## 8. 🎬 Mise en situation — Architecture d'une app avec agent IA

Reprenons le **chatbot RH** du notebook 03. Voici à quoi ressemble son architecture une fois en prod :

```
                                  🌍 Utilisateurs
                                       │
                            ┌──────────┴──────────┐
                            ▼                     ▼
                  ┌─────────────────┐    ┌─────────────────┐
                  │  Web (Next.js)  │    │  Teams Bot      │
                  │   Static + SSR  │    │  (Bot Framework)│
                  │   Azure App     │    │  Azure Bot      │
                  │   Service       │    │  Service        │
                  └─────────┬───────┘    └─────────┬───────┘
                            │                      │
                            └──────────┬───────────┘
                                       │   HTTPS / JWT (Entra ID)
                                       ▼
                          ┌──────────────────────────┐
                          │   Backend (FastAPI)      │
                          │   App Service Python     │
                          │                          │
                          │   /api/chat   → agent    │
                          │   /api/health → 200      │
                          │   /api/usage  → tokens   │
                          └─────────┬────────────────┘
                                    │
                ┌───────────────────┼──────────────────┐
                ▼                   ▼                  ▼
      ┌─────────────────┐  ┌────────────────┐  ┌──────────────────┐
      │  Foundry        │  │  Cosmos DB     │  │ Application      │
      │  (agent +       │  │  (historique   │  │ Insights         │
      │   index RAG)    │  │   conversations)│  │ (traces, métriques)
      └─────────────────┘  └────────────────┘  └──────────────────┘
             │
             ▼
      ┌─────────────────┐
      │  Azure AI Search│  ← index vectoriel pour le RAG
      │  Blob Storage   │  ← documents RH (PDF)
      └─────────────────┘
```

**Lecture rapide :**
- **Frontend** : deux clients (un web, un Teams).
- **Backend** : un service FastAPI qui expose une **API REST** (`/api/chat` notamment).
- **L'agent Foundry** est appelé **par le backend**, jamais directement par le frontend (sinon il faudrait exposer une clé / un token au client → mauvaise sécurité).
- L'**historique** des conversations est stocké dans **Cosmos DB**.
- Les **traces** (notebook 06) atterrissent dans Application Insights.
- Les **documents RH** sont en Blob Storage, indexés vectoriellement dans Azure AI Search → l'agent fait du RAG dessus.

**Streaming de la réponse LLM ?** Le backend ouvre un **SSE** ou **WebSocket** vers le frontend pour faire défiler les tokens au fur et à mesure.

## 9. Authentification & autorisation — *teaser du notebook 05*

Comment les clients prouvent leur identité à l'API ? Trois mécanismes courants :

| Mécanisme | Comment | Quand |
|-----------|---------|-------|
| **Session + cookie** | Le serveur crée une session, renvoie un cookie au navigateur. | Webs classiques, monolithes. |
| **JWT** *(JSON Web Token)* | Le serveur signe un token, le client l'envoie dans `Authorization: Bearer ...`. | APIs stateless, microservices. |
| **OAuth 2.0 / OIDC** | Standard pour déléguer l'auth à un fournisseur (Microsoft Entra ID, Google…). | Apps modernes, SSO entreprise. |

Dans le cloud Azure : on utilise **Entra ID** (OIDC) côté utilisateur + **Managed Identity** côté service-à-service. Détails dans le **notebook 05**.

> 🛑 **Ne stockez jamais un mot de passe en clair**. Hash avec **bcrypt** ou **Argon2**. Et idéalement, **déléguez l'authentification** à un fournisseur (Entra ID, Auth0, Clerk…) plutôt que de la rouler vous-même.

## 10. Récap — Cheatsheet

| Terme | Vite dit |
|-------|----------|
| **Client / Serveur** | Le navigateur envoie, le serveur répond. |
| **HTTP** | Le protocole. Méthodes + URL + headers + body + status. |
| **Frontend** | Ce que voit l'utilisateur. Tourne dans le navigateur. |
| **Backend** | La logique métier, l'accès aux données. Tourne sur un serveur. |
| **API** | L'interface qu'expose le backend. |
| **REST** | Style d'API basé sur les ressources et les verbes HTTP. |
| **OpenAPI** | Spec qui décrit une API REST. |
| **GraphQL / gRPC / SSE / WebSocket** | Alternatives à REST selon le besoin. |
| **JSON** | Le format d'échange par défaut. |
| **SPA** | Frontend qui se rend côté navigateur. |
| **SSR** | Frontend rendu côté serveur. |
| **SSG / Static** | Pages pré-générées, servies par CDN. |
| **Monolithe / Microservices / Serverless** | 3 grands styles backend. |
| **BFF** | *Backend For Frontend*, un backend par type de client. |
| **CDN** | Cache mondial pour les fichiers statiques. |
| **Reverse proxy** | Point d'entrée HTTPS qui route. |
| **Load balancer** | Distribue le trafic sur N instances. |
| **Cache** | Mémoriser temporairement (Redis). |
| **JWT / OIDC** | Tokens d'authentification. |
| **CORS** | Politique navigateur qui empêche le frontend A d'appeler l'API B sauf permission. |

**Suite :**
- Notebook 05 — **Sécurité cloud** : Entra ID, RBAC, Managed Identity, Key Vault.
- Notebook 06 — **Observabilité** : tracer un appel HTTP de bout en bout.

📚 Pour aller plus loin :
- *Architectural styles* (Microsoft Architecture Center) : https://learn.microsoft.com/azure/architecture/guide/architecture-styles/
- *Web API design best practices* : https://learn.microsoft.com/azure/architecture/best-practices/api-design
- *HTTP status codes* : https://developer.mozilla.org/docs/Web/HTTP/Status
- *MDN — Introduction au web* : https://developer.mozilla.org/fr/docs/Learn/